<a href="https://colab.research.google.com/github/24211a6792-PuneetKumarSoni/Deep-Learning--Programs/blob/main/Week_11%2C12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#week- 11-(i)
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Reshape, Flatten
from tensorflow.keras.datasets import mnist

# Load MNIST
(x, _), (_, _) = mnist.load_data()

# Normalize images to [-1, 1]
x = x.astype("float32") / 127.5 - 1
x = x.reshape(-1, 784)

# Generator
generator = Sequential([
    Dense(128, activation="relu", input_shape=(100,)),
    Dense(256, activation="relu"),
    Dense(784, activation="tanh"),
    Reshape((28, 28))
])

# Discriminator
discriminator = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(256, activation="relu"),
    Dense(128, activation="relu"),
    Dense(1, activation="sigmoid")
])

# Compile discriminator
discriminator.compile(
    optimizer="adam",
    loss="binary_crossentropy"
)

# Create GAN
discriminator.trainable = False

gan = Sequential([
    generator,
    discriminator
])

gan.compile(
    optimizer="adam",
    loss="binary_crossentropy"
)

# Training
for epoch in range(10):

    # Generate fake images
    noise = np.random.normal(0, 1, (64, 100))
    fake = generator.predict(noise, verbose=0)

    # Select real images
    real = x[np.random.randint(0, x.shape[0], 64)]
    real = real.reshape(-1, 28, 28)

    # Train discriminator
    discriminator.trainable = True

    discriminator.train_on_batch(
        real,
        np.ones((64, 1))
    )

    discriminator.train_on_batch(
        fake,
        np.zeros((64, 1))
    )

    # Train generator
    discriminator.trainable = False

    gan.train_on_batch(
        noise,
        np.ones((64, 1))
    )

    print("Epoch:", epoch + 1)

print("MLP GAN training completed")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch: 1
Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5
Epoch: 6
Epoch: 7
Epoch: 8
Epoch: 9
Epoch: 10
MLP GAN training completed


In [ ]:
# week -11-(ii)import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Reshape,
    Flatten,
    Conv2D,
    Conv2DTranspose
)

from tensorflow.keras.datasets import mnist

# Load MNIST
(x, _), (_, _) = mnist.load_data()

# Normalize images to [-1, 1]
x = x.astype("float32") / 127.5 - 1

# Add channel dimension
x = x.reshape(-1, 28, 28, 1)

# Generator
generator = Sequential([
    Dense(
        7 * 7 * 128,
        input_shape=(100,)
    ),

    Reshape((7, 7, 128)),

    Conv2DTranspose(
        64,
        4,
        strides=2,
        padding="same",
        activation="relu"
    ),

    Conv2DTranspose(
        1,
        4,
        strides=2,
        padding="same",
        activation="tanh"
    )
])

# Discriminator
discriminator = Sequential([
    Conv2D(
        64,
        4,
        strides=2,
        padding="same",
        activation="relu",
        input_shape=(28, 28, 1)
    ),

    Flatten(),

    Dense(
        1,
        activation="sigmoid"
    )
])

# Compile discriminator
discriminator.compile(
    optimizer="adam",
    loss="binary_crossentropy"
)

# GAN
discriminator.trainable = False

gan = Sequential([
    generator,
    discriminator
])

gan.compile(
    optimizer="adam",
    loss="binary_crossentropy"
)

# Training
for epoch in range(10):

    # Random noise
    noise = np.random.normal(
        0, 1, (64, 100)
    )

    # Generate fake images
    fake = generator.predict(
        noise,
        verbose=0
    )

    # Select real images
    real = x[
        np.random.randint(
            0,
            x.shape[0],
            64
        )
    ]

    # Train discriminator
    discriminator.trainable = True

    discriminator.train_on_batch(
        real,
        np.ones((64, 1))
    )

    discriminator.train_on_batch(
        fake,
        np.zeros((64, 1))
    )

    # Train generator
    discriminator.trainable = False

    gan.train_on_batch(
        noise,
        np.ones((64, 1))
    )

    print("Epoch:", epoch + 1)

print("Convolutional GAN training completed")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch: 1
Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5
Epoch: 6
Epoch: 7
Epoch: 8
Epoch: 9
Epoch: 10
Convolutional GAN training completed


In [ ]:
#week-12
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.datasets import mnist

# Load MNIST dataset
(x_train, _), (x_test, _) = mnist.load_data()

# Normalize the images
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Convert 28x28 images into 784-dimensional vectors
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# Input layer
input_img = Input(shape=(784,))

# Encoder
encoded = Dense(128, activation="relu")(input_img)
encoded = Dense(32, activation="relu")(encoded)

# Decoder
decoded = Dense(128, activation="relu")(encoded)
decoded = Dense(784, activation="sigmoid")(decoded)

# Create Autoencoder
autoencoder = Model(input_img, decoded)

# Compile
autoencoder.compile(
    optimizer="adam",
    loss="binary_crossentropy"
)

# Train
autoencoder.fit(
    x_train,
    x_train,
    epochs=5,
    batch_size=256,
    validation_data=(x_test, x_test)
)

# Reconstruct test images
reconstructed = autoencoder.predict(x_test)

print("Original shape:", x_test[0].shape)
print("Reconstructed shape:", reconstructed[0].shape)

print("Autoencoder training completed")

Epoch 1/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - loss: 0.2322 - val_loss: 0.1522
Epoch 2/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - loss: 0.1334 - val_loss: 0.1189
Epoch 3/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - loss: 0.1146 - val_loss: 0.1080
Epoch 4/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - loss: 0.1062 - val_loss: 0.1018
Epoch 5/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - loss: 0.1012 - val_loss: 0.0986
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Original shape: (784,)
Reconstructed shape: (784,)
Autoencoder training completed
